Use the compositional CLR data to find out which niches are correlated with each other and which cell types (like epithelial_luminal or macrophage) physically define them (20/04)

In [ ]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.stats import spearmanr

adata = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

# Strip var_names prefix
adata.var_names = [v.replace('meanscell_abundance_w_sf_', '')
                   for v in adata.var_names]

CLUSTER_KEY = 'hc_ward_k10'
OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/exploration_mean/"
           "niche_biology_nt14")
os.makedirs(OUT_DIR, exist_ok=True)

X = adata.X.toarray() if sp.issparse(adata.X) else adata.X
X_df = pd.DataFrame(X, columns=adata.var_names, index=adata.obs_names)
X_df['niche'] = adata.obs[CLUSTER_KEY].values

# Mean CLR profile per niche — used throughout
niche_profiles = X_df.groupby('niche').mean()
print("Niche profiles shape:", niche_profiles.shape)
print(niche_profiles)

Cluster-to-cluster correlation. This answers: which niches are biologically similar to each other?


In [ ]:
# Pearson correlation between niche mean CLR profiles
corr_matrix = niche_profiles.T.corr(method='pearson')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    cmap='coolwarm', center=0,
    vmin=-1, vmax=1,
    annot=True, fmt='.2f',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Pearson correlation'}
)
ax.set_title('Niche-to-niche correlation\n'
             'Based on mean CLR cell type profiles (HC Ward k=10)',
             fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'niche_niche_correlation.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# For each niche, correlate its spot-level CLR values 
# with each cell type across all spots
# Result: 10 niches x 22 cell types

niche_ct_corr = pd.DataFrame(
    index=sorted(adata.obs[CLUSTER_KEY].unique()),
    columns=adata.var_names,
    dtype=float
)

for niche in niche_ct_corr.index:
    mask = adata.obs[CLUSTER_KEY] == niche
    # For each cell type, correlation between 
    # being in this niche (1/0) and cell type abundance
    niche_binary = mask.astype(int).values
    for ct in adata.var_names:
        ct_values = X_df[ct].values
        r, _ = spearmanr(niche_binary, ct_values)
        niche_ct_corr.loc[niche, ct] = r

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    niche_ct_corr.astype(float),
    cmap='RdBu_r', center=0,
    annot=True, fmt='.2f',
    linewidths=0.3,
    ax=ax,
    cbar_kws={'label': 'Spearman r'}
)
ax.set_title('Niche-to-cell-type correlation\n'
             'Spearman r between niche membership and CLR abundance',
             fontsize=12)
ax.set_xlabel('Cell type')
ax.set_ylabel('Niche')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'niche_celltype_correlation.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Print the exact correlation values so you have ground truth
print("=== Niche-to-niche correlation matrix ===")
print(corr_matrix.round(2))

print("\n=== Niche-to-cell-type correlation matrix ===")
print(niche_ct_corr.astype(float).round(2))

In [ ]:
# Plot spatial niches for a representative subset of slides
slides_to_plot = [
    'Patient_1_H1_2',   # Erickson cancer
    'Patient_2_H3_1',   # Erickson cancer
    'HP09_CZ_ST',       # Hu normal CZ
    'HP05_PZ_ST',       # Hu normal PZ
]

for slide_id in slides_to_plot:
    mask = adata.obs['slide_id'] == slide_id
    adata_slide = adata[mask].copy()
    
    if adata_slide.n_obs == 0:
        print(f"Skipping {slide_id} — no spots found")
        continue
    
    sc.pl.spatial(
        adata_slide,
        color=CLUSTER_KEY,
        library_id=slide_id,
        spot_size=1.5,
        title=f'{slide_id}\nHC Ward k=10 niches',
        save=f'_spatial_{slide_id}.png',
        show=False
    )
    print(f"Saved spatial plot for {slide_id}")

Cell 4 — Spatial overlays on H&E
Project niche labels onto the actual tissue images — one plot per slide

To overlay the H&E we need to know how the spatial coordinates relate to the image pixels — they need to be on the same scale

In [ ]:
from PIL import Image
import numpy as np

# Check image size
img_path = "/path/to/data/Erickson_2022_Data/svw96g68dv-1/Histological_images/Patient_1/Visium/H1_2.jpg"
img = Image.open(img_path)
print("Image size (width x height):", img.size)
print("Image mode:", img.mode)

# Check spatial coordinates for this slide
mask = adata.obs['slide_id'] == 'Patient_1_H1_2'
coords = adata[mask].obsm['spatial']
print("\nSpatial coords shape:", coords.shape)
print("X range:", coords[:, 0].min(), "to", coords[:, 0].max())
print("Y range:", coords[:, 1].min(), "to", coords[:, 1].max())

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

Image.MAX_IMAGE_PIXELS = None  # suppress decompression warning

img_path = ("/path/to/data/"
            "Erickson_2022_Data/svw96g68dv-1/Histological_images/"
            "Patient_1/Visium/H1_2.jpg")

# ── Load and downsample image ────────────────────────────────────────
img = Image.open(img_path)
img.thumbnail((4000, 4000), Image.LANCZOS)  # downsample for memory
img_arr = np.array(img)

# Compute scale factor applied by thumbnail
scale = img_arr.shape[1] / 13184  # new_width / original_width
print(f"Scale factor: {scale:.4f}")

# ── Get slide data ───────────────────────────────────────────────────
slide_id = 'Patient_1_H1_2'
mask = adata.obs['slide_id'] == slide_id
adata_slide = adata[mask]
coords = adata_slide.obsm['spatial'].copy()

# Scale coordinates to match downsampled image
coords_scaled = coords * scale

# ── Colour by niche ──────────────────────────────────────────────────
niches = sorted(adata.obs[CLUSTER_KEY].unique())
n_niches = len(niches)
cmap = cm.get_cmap('tab10', n_niches)
colour_map = {niche: cmap(i) for i, niche in enumerate(niches)}
labels = adata_slide.obs[CLUSTER_KEY].values
colours = [colour_map[l] for l in labels]

# ── Plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Left: H&E only
axes[0].imshow(img_arr)
axes[0].set_title(f'{slide_id}\nH&E only', fontsize=12)
axes[0].axis('off')

# Right: H&E + niche overlay
axes[1].imshow(img_arr)
axes[1].scatter(
    coords_scaled[:, 0],
    coords_scaled[:, 1],
    c=colours, s=8, alpha=0.7,
    rasterized=True, linewidths=0
)

# Legend
handles = [plt.scatter([], [], c=[colour_map[n]], s=30, label=f'Niche {n}')
           for n in niches]
axes[1].legend(handles=handles, bbox_to_anchor=(1.01, 1),
               loc='upper left', fontsize=9, title='HC Ward k=10')
axes[1].set_title(f'{slide_id}\nH&E + niche overlay', fontsize=12)
axes[1].axis('off')

plt.suptitle(f'Spatial niche map — {slide_id}\nErickson 2022, Patient 1',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'HE_overlay_{slide_id}.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: HE_overlay_{slide_id}.png")